In [8]:
import json
import os
import subprocess
import itertools
from datetime import datetime
import pandas as pd
from pathlib import Path
import sys
import pickle as pl
import torch
import itertools

In [9]:
env = os.environ.copy()
env['MKL_SERVICE_FORCE_INTEL'] = '1'
env['MKL_THREADING_LAYER'] = 'GNU'
env['OMP_NUM_THREADS'] = '1'

In [10]:
# parameter_grid_1 = {
#     "n_total": [5000],
#     "n_finetune": [2500],
#     "model_name": ["bert-base-uncased"],
#     "max_length": [256],
#     "num_labels": [6],
#     "batch_size": [32],
#     "learning_rate": [1e-2,0.003],
#     "num_epochs": [2],
#     "K": [15],
#     "lambda_min": [0.05],
#     "lambda_max": [0.95],
#     "interpolation": ["linear"],
#     "optimizer": ["Adam"],
#     "dataset": ["emotion"],
#     "proportionArr": [[0.15,0.15,0.3,0.35,0.05]]
# }

In [11]:
computed_vectors = [[0.3, 0.25, 0.45],
[0.3, 0.3, 0.4],
[0.3, 0.35, 0.35] ,
[0.3, 0.4, 0.3]  ,
[0.3, 0.45, 0.25] ,
[0.25, 0.3, 0.45]  ,
[0.35, 0.3, 0.35], 
[0.4, 0.3, 0.3] ,
[0.45, 0.3, 0.25] ,
[0.25, 0.45, 0.3]  ,
[0.35, 0.35, 0.3]  ,
[0.45, 0.25, 0.3]]

In [12]:
step = 0.05
fixed_value = 0.3  # One class fixed at this
remaining_sum = 1.0 - fixed_value  # = 0.7 to distribute among other 2 classes
max_value = 0.55
min_value = 0.15

# Generate values from 0.05 to 0.65
values = [round(step * i, 2) for i in range(int(min_value/step), int(max_value/step) + 1)]

vectors = []
seen = set()  # Track unique combinations using sorted tuples
num_classes = 3
indices = list(range(num_classes))

for fixed_idx in indices:  # Which position to fix at 0.3
    for x in values:
        y = round(remaining_sum - x, 2)
        
        # Only include if y is valid (0.05 <= y <= 0.65)
        if min_value <= y <= max_value:
            v = [0.0] * num_classes
            v[fixed_idx] = fixed_value
            
            # Fill other two positions
            other_indices = [idx for idx in indices if idx != fixed_idx]
            v[other_indices[0]] = x
            v[other_indices[1]] = y
            
            # Check if this combination (in sorted form) already exists
            v = tuple(v)
            if v not in seen and list(v) not in computed_vectors:
                seen.add(v)
                vectors.append(v)

print(f"Total unique combinations: {len(vectors)}")
for i, v in enumerate(vectors, 1):
    print(f"{i:2d}. {v}  (sum = {sum(v):.2f})")

Total unique combinations: 12
 1. (0.3, 0.15, 0.55)  (sum = 1.00)
 2. (0.3, 0.2, 0.5)  (sum = 1.00)
 3. (0.3, 0.5, 0.2)  (sum = 1.00)
 4. (0.3, 0.55, 0.15)  (sum = 1.00)
 5. (0.15, 0.3, 0.55)  (sum = 1.00)
 6. (0.2, 0.3, 0.5)  (sum = 1.00)
 7. (0.5, 0.3, 0.2)  (sum = 1.00)
 8. (0.55, 0.3, 0.15)  (sum = 1.00)
 9. (0.15, 0.55, 0.3)  (sum = 1.00)
10. (0.2, 0.5, 0.3)  (sum = 1.00)
11. (0.5, 0.2, 0.3)  (sum = 1.00)
12. (0.55, 0.15, 0.3)  (sum = 1.00)


In [13]:
vectors

[(0.3, 0.15, 0.55),
 (0.3, 0.2, 0.5),
 (0.3, 0.5, 0.2),
 (0.3, 0.55, 0.15),
 (0.15, 0.3, 0.55),
 (0.2, 0.3, 0.5),
 (0.5, 0.3, 0.2),
 (0.55, 0.3, 0.15),
 (0.15, 0.55, 0.3),
 (0.2, 0.5, 0.3),
 (0.5, 0.2, 0.3),
 (0.55, 0.15, 0.3)]

In [14]:
# 0.000006 - stable learning rate

parameter_grid_2 = {
    "n_total": [5000],
    "n_finetune": [2500],
    "model_name": ["bert-base-uncased"],
    "max_length": [256],
    "num_labels": [3],
    "batch_size": [32],
    "learning_rate": [0.000006],
    "num_epochs": [2],
    "K": [15],
    "lambda_min": [0.05],
    "lambda_max": [0.95],
    "interpolation": ["slerp","ties","linear","model_baseline"],
    "optimizer": ["Adam"],
    "dataset": ["snli"],
    "proportionArr": vectors
}

# 0.000006, 0.00001

# # to be fixed
# [0.1,0.7,0.1,0.1] 

# # done 
# [0.25,0.25,0.25,0.25]
# [0.3,0.1,0.3,0.3]


    # "learning_rate": [0.000006,0.000003],



In [15]:
# parameter_grid_3 = {
#     "n_total": [5000],
#     "n_finetune": [2500],
#     "model_name": ["bert-base-uncased"],
#     "max_length": [256],
#     "num_labels": [5],
#     "batch_size": [32],
#     "learning_rate": [0.001,0.003],
#     "num_epochs": [2],
#     "K": [15],
#     "lambda_min": [0.05],
#     "lambda_max": [0.95],
#     "interpolation": ["linear"],
#     "optimizer": ["Adam"],
#     "dataset": ["yelp_review_full"],
#     "proportionArr": [[0.27,0.1,0.27,0.1,0.26],]
    
    
    # # to be fixed
    # [0.1,0.1,0.6,0.1,0.1]
    # [0.05,0.1,0.05,0.7,0.1]
    
    
    # # done 
    # [0.2,0.2,0.2,0.2,0.2]


In [16]:
parameter_grids = []
parameter_grids.append(parameter_grid_2)
# parameter_grids.append(parameter_grid_3)

In [17]:
# Or define specific combinations
specific_configs = []

In [18]:
def generate_all_combinations(param_grid):
    keys = param_grid.keys()
    values = param_grid.values()
    combinations = []
    
    for combination in itertools.product(*values):
        config = dict(zip(keys, combination))
        combinations.append(config)
    
    return combinations

def create_config_file(config, experiment_path):
    config['experiment_name'] = experiment_path
    config_path = experiment_path + ".json"
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=4)
    print(f"Created config file: {config_path}")

def run_experiment(config, experiment_name):
    try:
        # Create config file
        create_config_file(config, f"{experiment_name}")
        
        print(f"\nRunning experiment: {experiment_name}")
        print(f"Config: {config}")
        
        # result = subprocess.run([sys.executable, "agnews_sample_hacking_last_layer.py", f"{experiment_name}.json"], capture_output=True, text=True,shell=True)
        result = subprocess.call([sys.executable, "agnews_sample_hacking_last_layer.py", f"{experiment_name}.json"],env=env)
        
        if result == 0:
            print(f"✅ Experiment {experiment_name} completed successfully")
        else:
            print(f"❌ Experiment {experiment_name} failed")
            # print("Error:", result.stderr)
        
        return {
            "experiment_name": experiment_name,
            "config": config,
            "success": result == 0,
            # "stdout": result.stdout,
            # "stderr": result.stderr
        }
        
    except Exception as e:
        print(f"❌ Exception in {experiment_name}: {str(e)}")
        return {
            "experiment_name": experiment_name,
            "config": config,
            "success": False,
            "error": str(e)
        }

In [19]:
configs_to_run = generate_all_combinations(parameter_grids[0])
len(configs_to_run)

48

In [ ]:
# Choose experiment mode
USE_GRID_SEARCH = True # Set to True for grid search, False for specific configs

if USE_GRID_SEARCH:
    for parameter_grid in parameter_grids:
        configs_to_run = generate_all_combinations(parameter_grid)
        print(f"Total configurations to run: {len(configs_to_run)}")
        
        # Run all experiments
        results = []
        for i, config in enumerate(configs_to_run):
            experiment_name = f"{config['dataset']}_{config['interpolation']}_{config['proportionArr']}"
            result = run_experiment(config, experiment_name)
            results.append(result)

        # Summary
        successful = sum(1 for r in results if r["success"])
        print(f"\n{'='*50}")
        print(f"EXPERIMENT SUMMARY")
        print(f"{'='*50}")
        print(f"Total experiments: {len(results)}")
        print(f"Successful: {successful}")
        print(f"Failed: {len(results) - successful}")
        
else:
    configs_to_run = specific_configs

Total configurations to run: 48
Created config file: snli_slerp_(0.3, 0.15, 0.55).json

Running experiment: snli_slerp_(0.3, 0.15, 0.55)
Config: {'n_total': 5000, 'n_finetune': 2500, 'model_name': 'bert-base-uncased', 'max_length': 256, 'num_labels': 3, 'batch_size': 32, 'learning_rate': 6e-06, 'num_epochs': 2, 'K': 15, 'lambda_min': 0.05, 'lambda_max': 0.95, 'interpolation': 'slerp', 'optimizer': 'Adam', 'dataset': 'snli', 'proportionArr': (0.3, 0.15, 0.55), 'experiment_name': 'snli_slerp_(0.3, 0.15, 0.55)'}


2026-01-19 08:08:59.149067: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-19 08:08:59.620223: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-19 08:08:59.620267: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-19 08:08:59.699200: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-19 08:08:59.863556: I tensorflow/core/platform/cpu_feature_guar

✓ Mergekit imported successfully
Loading configuration from: snli_slerp_(0.3, 0.15, 0.55).json
✓ Configuration loaded successfully

Output directory created: snli_slerp_(0.3, 0.15, 0.55)

STEP 1: Loading the snli Dataset
Full AG News training set size: 550152
Dataset features: {'premise': Value('string'), 'hypothesis': Value('string'), 'label': ClassLabel(names=['entailment', 'neutral', 'contradiction'])}
Total samples in dataset: 550152
Valid samples (label != -1): 549367
Selected subset D with 5000 samples
Label 0....samples needed 750.....needed proportion: 0.3....actual proportion: 0.3
Label 1....samples needed 375.....needed proportion: 0.15....actual proportion: 0.15
Label 2....samples needed 1375.....needed proportion: 0.55....actual proportion: 0.55
Size of the computed finetuning set: 2500 
Fine-tuning set expected size: 2500
Size of the fixed/updated finetuning set: 2500 
Selected fine-tuning subset D' with 2500 samples

Dataset Statistics:
  Total samples |D|: 5000
  Fine-tu

/home/aditya/miniconda3/envs/myenv/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Epoch 1/2:  62%|██████▏   | 49/79 [00:45<00:11,  2.56it/s, loss=1.02] 

In [ ]:
# When a parent process starts a child process via subprocess, the two are separate entities with their own memory space.
# The parent process is not notified in real-time about the filesystem modifications the child process is making.